PAra El Consumo

In [0]:
%pip install yellowbrick 

In [0]:
# Databricks notebook: gold_analytics_olist
# ------------------------------------------------------------
# Librerías
# ------------------------------------------------------------
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
from pyspark.sql import functions as F

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
CORTE = pd.to_datetime("2018-08-31")   # Fecha límite del dataset


In [0]:
# ------------------------------------------------------------
# Configuración del entorno (Unity Catalog)
# ------------------------------------------------------------
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

# ✅ Ajusta esto según tu catálogo UC (usualmente "main" o el que te haya dado tu admin)
CATALOG = "workspace"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# Selecciona el catálogo y el esquema de trabajo
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
spark.sql(f"USE {CATALOG}.{SCHEMA_SILVER}")

# ------------------------------------------------------------
# Cargar datos desde Silver (Unity Catalog)
# ------------------------------------------------------------
orders = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.olist_orders_dataset").toPandas()
items = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.olist_order_items_dataset").toPandas()
customers = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.olist_customers_dataset").toPandas()

# ------------------------------------------------------------
# Filtrar solo pedidos entregados
# ------------------------------------------------------------
orders = orders[orders["order_status"] == "delivered"]

# ------------------------------------------------------------
# Unión: items + orders
# ------------------------------------------------------------
df = items.merge(
    orders[["order_id", "customer_id", "order_purchase_timestamp"]],
    on="order_id", how="inner"
)

# ------------------------------------------------------------
# Procesamiento de fechas y valores
# ------------------------------------------------------------
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["total"] = df["price"] + df["freight_value"]

print(f"✅ Dataset fusionado: {df.shape[0]} filas, {df.shape[1]} columnas")

# ------------------------------------------------------------
# Guardar en GOLD (Unity Catalog)
# ------------------------------------------------------------
df_spark = spark.createDataFrame(df)
df_spark.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.olist_orders_gold")

print(f"✅ Datos guardados correctamente en {CATALOG}.{SCHEMA_GOLD}.olist_orders_gold")


In [0]:
# ------------------------------------------------------------
# Productos más vendidos
# ------------------------------------------------------------
top_products = (
    df.groupby("product_id")
      .agg(total_ventas=("total", "sum"),
           cantidad_vendida=("order_id", "count"))
      .reset_index()
      .sort_values(by="cantidad_vendida", ascending=False)
)
# Crear schema GOLD
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
# Guardar como tabla GOLD
spark.createDataFrame(top_products).write.mode("overwrite").format("delta").saveAsTable("gold.top_productos")

print("✅ Tabla creada: gold.top_productos")

# Visualización top 10
plt.figure(figsize=(10,5))
sns.barplot(
    data=top_products.head(10),
    x="cantidad_vendida", y="product_id", palette="viridis"
)
plt.title("🔝 Top 10 productos más vendidos")
plt.xlabel("Cantidad vendida")
plt.ylabel("ID de producto")
plt.show()


In [0]:
# ------------------------------------------------------------
# Cálculo RFM
# ------------------------------------------------------------
rfm = (df.groupby("customer_id")
         .agg(
             last_purchase=("order_purchase_timestamp", "max"),
             frequency=("order_id", "nunique"),
             monetary=("total", "sum"))
       .reset_index())

rfm["recency"] = (CORTE - rfm["last_purchase"]).dt.days
rfm = rfm[["customer_id", "recency", "frequency", "monetary"]]


In [0]:
# ------------------------------------------------------------
# Clustering K-Means (segmentación)
# ------------------------------------------------------------
X = rfm[["recency", "frequency", "monetary"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Determinar número óptimo de clusters
model = KMeans(random_state=42, n_init="auto")
visualizer = KElbowVisualizer(model, k=(2, 10))
visualizer.fit(X_scaled)
visualizer.show()

# Aplicar K-Means con k óptimo
k_opt = visualizer.elbow_value_
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init="auto")
rfm["segmento"] = kmeans.fit_predict(X_scaled)

# Guardar tabla GOLD
spark.createDataFrame(rfm).write.mode("overwrite").format("delta").saveAsTable("gold.segmentacion_clientes")
print("✅ Tabla creada: gold.segmentacion_clientes")


In [0]:
# ------------------------------------------------------------
# Estadísticas por segmento
# ------------------------------------------------------------
resumen = (rfm.groupby("segmento")
             .agg(
                 clientes=("customer_id", "count"),
                 recency_med=("recency", "median"),
                 freq_med=("frequency", "median"),
                 monetary_med=("monetary", "median"))
           .round(2)
           .reset_index())

spark.createDataFrame(resumen).write.mode("overwrite").format("delta").saveAsTable("gold.resumen_segmentos")

print("✅ Tabla creada: gold.resumen_segmentos")
print(resumen)
